# 神经网络超参数调整 (PyTorch版)

> 本教程是 [超参数调整 (TensorFlow版)](./超参数调整.ipynb) 的 PyTorch 等价版本。

本教程介绍神经网络超参数调整的系统方法，包括学习率查找器、随机搜索和1-Cycle学习率策略。
PyTorch 拥有内置的 `torch.optim.lr_scheduler.OneCycleLR`，这是相比 Keras 的一个重要优势——
无需自己实现1-Cycle调度器，直接调用即可。

## 学习目标

1. 理解超参数对模型性能的影响
2. 手动实现学习率查找器(LR Finder)，理解其工作原理
3. 学会使用随机搜索进行超参数优化
4. 掌握 PyTorch 内置 `OneCycleLR` 调度器的使用
5. 对比固定学习率与1-Cycle策略的训练效果

## 主要超参数

| 超参数 | 影响 | 典型范围 |
|--------|------|----------|
| 学习率 | 收敛速度和稳定性 | 1e-4 ~ 1e-1 |
| 隐藏层数 | 模型复杂度 | 1 ~ 5 |
| 神经元数量 | 层容量 | 32 ~ 512 |
| 批量大小 | 训练稳定性 | 16 ~ 256 |
| 激活函数 | 非线性特性 | relu, tanh, elu |

## 1. 环境配置与数据准备

In [ ]:
import copy
import os

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

# 设置随机种子 / Set random seed
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

print(f"PyTorch版本 / PyTorch version: {torch.__version__}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"设备 / Device: {device}")

In [ ]:
# 加载数据 / Load data
housing = fetch_california_housing()

# 划分数据集 / Split dataset
X_train_full, X_test, y_train_full, y_test = train_test_split(
    housing.data, housing.target, test_size=0.2, random_state=RANDOM_SEED
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, test_size=0.25, random_state=RANDOM_SEED
)

# 标准化 / Standardize
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_valid = scaler.transform(X_valid)
X_test = scaler.transform(X_test)

# 转换为PyTorch张量 / Convert to PyTorch tensors
X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.FloatTensor(y_train).unsqueeze(1)
X_valid_t = torch.FloatTensor(X_valid)
y_valid_t = torch.FloatTensor(y_valid).unsqueeze(1)
X_test_t = torch.FloatTensor(X_test)
y_test_t = torch.FloatTensor(y_test).unsqueeze(1)

# 创建DataLoader / Create DataLoaders
BATCH_SIZE = 32
train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

valid_dataset = TensorDataset(X_valid_t, y_valid_t)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)

test_dataset = TensorDataset(X_test_t, y_test_t)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"训练集 / Training set: {X_train.shape[0]} 样本 / samples")
print(f"验证集 / Validation set: {X_valid.shape[0]} 样本 / samples")
print(f"测试集 / Test set: {X_test.shape[0]} 样本 / samples")
print(f"每epoch步数 / Steps per epoch: {len(train_loader)}")

## 2. 模型构建函数

定义一个参数化的模型构建函数，便于超参数搜索。PyTorch 中使用工厂函数返回 `nn.Module`，
等价于 Keras 中返回编译好的 `keras.Model`。

In [ ]:
def build_model(n_hidden=1, n_neurons=30, learning_rate=0.01,
                activation='relu', input_dim=8):
    """
    构建可配置的回归模型 / Build a configurable regression model.

    返回一个 nn.Module 实例和对应的优化器。
    等价于 Keras 版的 build_model()，但 PyTorch 需要分别返回模型和优化器。

    Returns an nn.Module instance and its optimizer.
    Equivalent to the Keras build_model(), but PyTorch requires
    returning the model and optimizer separately.

    Parameters:
    -----------
    n_hidden : int
        隐藏层数量 / Number of hidden layers
    n_neurons : int
        每层神经元数量 / Number of neurons per hidden layer
    learning_rate : float
        学习率 / Learning rate
    activation : str
        激活函数类型 / Activation function type
    input_dim : int
        输入特征维度 / Input feature dimension

    Returns:
    --------
    tuple : (model, optimizer) 模型和优化器 / model and optimizer
    """
    # 激活函数映射 / Activation function mapping
    activation_map = {
        'relu': nn.ReLU,
        'tanh': nn.Tanh,
        'elu': nn.ELU,
        'leaky_relu': nn.LeakyReLU,
    }
    act_fn = activation_map.get(activation, nn.ReLU)

    # 构建层列表 / Build layer list
    layers = []
    in_dim = input_dim
    for _ in range(n_hidden):
        layers.append(nn.Linear(in_dim, n_neurons))
        layers.append(act_fn())
        in_dim = n_neurons
    # 输出层（回归任务不需要激活函数）/ Output layer (no activation for regression)
    layers.append(nn.Linear(in_dim, 1))

    model = nn.Sequential(*layers).to(device)
    optimizer = optim.SGD(model.parameters(), lr=learning_rate)

    return model, optimizer


# 测试模型构建 / Test model building
test_model, test_optimizer = build_model(n_hidden=2, n_neurons=30)
print(test_model)
print(f"\n参数数量 / Number of parameters: {sum(p.numel() for p in test_model.parameters()):,}")

## 3. 学习率查找器 (LR Finder)

学习率是最重要的超参数。LR Finder 通过指数增长学习率来找到最佳范围。

### 原理

1. 从极小学习率开始（如1e-10）
2. 每个batch后指数增加学习率
3. 记录学习率和损失的对应关系
4. 选择损失快速下降区域的学习率

### PyTorch 实现要点

Keras 中 LR Finder 是一个 `Callback`，通过 `on_train_batch_end` 自动更新学习率。
PyTorch 没有内置回调系统，我们需要手动实现训练循环，在每个 batch 后通过
`optimizer.param_groups[0]['lr']` 来读取和修改学习率。这是一个很好的教学练习！

In [ ]:
class LRFinder:
    """
    学习率查找器 / Learning Rate Finder.

    通过指数增长学习率，找到最佳学习率范围。
    PyTorch 版本需要手动实现训练循环，在每个 batch 后更新学习率。

    Finds the optimal learning rate range by exponentially increasing
    the learning rate. The PyTorch version requires a manual training
    loop that updates the learning rate after each batch.

    Parameters:
    -----------
    model : nn.Module
        待测试的模型 / Model to test
    optimizer : optim.Optimizer
        优化器 / Optimizer
    criterion : nn.Module
        损失函数 / Loss function
    min_lr : float
        最小学习率 / Minimum learning rate
    max_lr : float
        最大学习率 / Maximum learning rate
    steps : int
        总测试步数 / Total number of test steps
    """

    def __init__(self, model, optimizer, criterion,
                 min_lr=1e-10, max_lr=10.0, steps=100):
        self.model = model
        self.optimizer = optimizer
        self.criterion = criterion
        self.min_lr = min_lr
        self.max_lr = max_lr
        self.steps = steps
        # 计算每步的乘法因子 / Calculate multiplicative factor per step
        self.factor = np.exp(np.log(max_lr / min_lr) / steps)
        self.lrs = []
        self.losses = []
        self.best_loss = float('inf')

    def run(self, train_loader, device='cpu'):
        """
        运行LR Finder / Run the LR Finder.

        遍历训练数据，逐步增加学习率并记录损失。
        当损失爆炸（超过最佳损失的4倍）时自动停止。

        Iterates through training data, gradually increasing the
        learning rate and recording the loss. Stops automatically
        when the loss explodes (exceeds 4x the best loss).

        Parameters:
        -----------
        train_loader : DataLoader
            训练数据加载器 / Training data loader
        device : str
            计算设备 / Compute device
        """
        # 保存初始模型状态以便恢复 / Save initial state for restoration
        initial_state = copy.deepcopy(self.model.state_dict())
        initial_lr = self.optimizer.param_groups[0]['lr']

        # 设置初始学习率 / Set initial learning rate
        self._set_lr(self.min_lr)

        self.model.train()
        step = 0
        stop = False

        for X_batch, y_batch in train_loader:
            if stop or step >= self.steps:
                break

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            self.optimizer.zero_grad()
            outputs = self.model(X_batch)
            loss = self.criterion(outputs, y_batch)
            loss.backward()
            self.optimizer.step()

            # 记录当前学习率和损失 / Record current LR and loss
            current_lr = self.optimizer.param_groups[0]['lr']
            current_loss = loss.item()
            self.lrs.append(current_lr)
            self.losses.append(current_loss)

            # 检查是否应该停止（损失爆炸）/ Check if should stop (loss explosion)
            if current_loss < self.best_loss:
                self.best_loss = current_loss
            if current_loss > self.best_loss * 4:
                stop = True
                break

            # 指数增加学习率 / Exponentially increase learning rate
            new_lr = current_lr * self.factor
            if new_lr > self.max_lr:
                stop = True
            else:
                self._set_lr(new_lr)

            step += 1

        # 恢复初始模型状态 / Restore initial model state
        self.model.load_state_dict(initial_state)
        self._set_lr(initial_lr)

    def _set_lr(self, lr):
        """设置优化器学习率 / Set optimizer learning rate."""
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = lr

    def plot(self, skip_start=10, skip_end=5):
        """
        绘制学习率-损失曲线 / Plot learning rate vs. loss curve.

        Parameters:
        -----------
        skip_start : int
            跳过开头的不稳定点 / Skip initial unstable points
        skip_end : int
            跳过结尾的爆炸点 / Skip final exploding points

        Returns:
        --------
        float : 建议学习率 / Suggested learning rate
        """
        lrs = self.lrs[skip_start:-skip_end] if skip_end > 0 else self.lrs[skip_start:]
        losses = self.losses[skip_start:-skip_end] if skip_end > 0 else self.losses[skip_start:]

        plt.figure(figsize=(10, 5))
        plt.plot(lrs, losses)
        plt.xscale('log')
        plt.xlabel('Learning Rate (log scale)')
        plt.ylabel('Loss')
        plt.title('Learning Rate Finder')
        plt.grid(True, alpha=0.3)

        # 标记最小损失点 / Mark minimum loss point
        min_idx = np.argmin(losses)
        plt.axvline(x=lrs[min_idx], color='r', linestyle='--',
                    label=f'Min Loss at LR={lrs[min_idx]:.2e}')
        plt.legend()
        plt.show()

        # 推荐学习率（通常取最小损失对应学习率的1/10）/ Suggest LR
        suggested_lr = lrs[min_idx] / 10
        print(f"最小损失对应学习率 / LR at min loss: {lrs[min_idx]:.2e}")
        print(f"建议使用学习率 / Suggested LR: {suggested_lr:.2e}")
        return suggested_lr


print("LR Finder 定义完成 / LR Finder defined")

In [ ]:
# 使用LR Finder / Use LR Finder
model, optimizer = build_model(n_hidden=1, n_neurons=30, learning_rate=1e-10)
criterion = nn.MSELoss()

# 计算步数（约1个epoch）/ Calculate steps (approximately 1 epoch)
n_steps = len(train_loader)

lr_finder = LRFinder(
    model, optimizer, criterion,
    min_lr=1e-10, max_lr=10.0, steps=n_steps
)

# 运行LR Finder / Run LR Finder
lr_finder.run(train_loader, device=device)

# 绘制结果并获取建议学习率 / Plot results and get suggested LR
suggested_lr = lr_finder.plot()

## 4. 训练工具函数

在超参数搜索之前，先定义通用的训练和评估函数。这些函数将在后续的随机搜索和对比实验中复用。

In [ ]:
def train_one_epoch(model, train_loader, criterion, optimizer, device):
    """
    训练一个epoch / Train for one epoch.

    Parameters:
    -----------
    model : nn.Module
        PyTorch模型 / PyTorch model
    train_loader : DataLoader
        训练数据加载器 / Training data loader
    criterion : loss function
        损失函数 / Loss function
    optimizer : optim.Optimizer
        优化器 / Optimizer
    device : torch.device
        计算设备 / Compute device

    Returns:
    --------
    float : 平均训练损失 / Average training loss
    """
    model.train()
    total_loss = 0.0
    total_samples = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * X_batch.size(0)
        total_samples += X_batch.size(0)
    return total_loss / total_samples


def evaluate(model, data_loader, criterion, device):
    """
    评估模型 / Evaluate the model.

    Parameters:
    -----------
    model : nn.Module
        PyTorch模型 / PyTorch model
    data_loader : DataLoader
        数据加载器 / Data loader
    criterion : loss function
        损失函数 / Loss function
    device : torch.device
        计算设备 / Compute device

    Returns:
    --------
    tuple : (平均MSE损失, 平均MAE) / (average MSE loss, average MAE)
    """
    model.eval()
    total_loss = 0.0
    total_mae = 0.0
    total_samples = 0
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            mae = nn.L1Loss()(y_pred, y_batch)
            total_loss += loss.item() * X_batch.size(0)
            total_mae += mae.item() * X_batch.size(0)
            total_samples += X_batch.size(0)
    return total_loss / total_samples, total_mae / total_samples


def fit(model, train_loader, valid_loader, criterion, optimizer,
        epochs, device, callbacks=None, lr_scheduler=None):
    """
    带回调和调度器的训练循环 / Training loop with callbacks and scheduler.

    等价于 Keras 的 model.fit()，集成了早停、检查点和学习率调度。

    Equivalent to Keras model.fit(), integrating early stopping,
    checkpointing, and learning rate scheduling.

    Parameters:
    -----------
    model : nn.Module
        PyTorch模型 / PyTorch model
    train_loader : DataLoader
        训练数据加载器 / Training data loader
    valid_loader : DataLoader
        验证数据加载器 / Validation data loader
    criterion : loss function
        损失函数 / Loss function
    optimizer : optim.Optimizer
        优化器 / Optimizer
    epochs : int
        最大训练轮数 / Maximum number of epochs
    device : torch.device
        计算设备 / Compute device
    callbacks : list, optional
        回调列表 / List of callbacks
    lr_scheduler : lr_scheduler, optional
        学习率调度器 / Learning rate scheduler

    Returns:
    --------
    dict : 训练历史 / Training history
    """
    history = {
        'loss': [], 'val_loss': [],
        'mae': [], 'val_mae': [],
        'lr': []
    }

    # 初始化回调 / Initialize callbacks
    if callbacks:
        for cb in callbacks:
            if hasattr(cb, 'on_train_begin'):
                cb.on_train_begin(logs=history)

    stop_training = False
    for epoch in range(epochs):
        if stop_training:
            break

        # 训练 / Train
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)

        # 评估 / Evaluate
        val_loss, val_mae = evaluate(model, valid_loader, criterion, device)
        _, train_mae = evaluate(model, train_loader, criterion, device)

        # 记录当前学习率 / Record current learning rate
        current_lr = optimizer.param_groups[0]['lr']

        # 保存历史 / Save history
        history['loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['mae'].append(train_mae)
        history['val_mae'].append(val_mae)
        history['lr'].append(current_lr)

        # 打印进度 / Print progress
        print(f"Epoch {epoch+1}/{epochs} - "
              f"loss: {train_loss:.4f} - mae: {train_mae:.4f} - "
              f"val_loss: {val_loss:.4f} - val_mae: {val_mae:.4f} - "
              f"lr: {current_lr:.6f}")

        # 学习率调度 / Learning rate scheduling
        if lr_scheduler is not None:
            if isinstance(lr_scheduler, optim.lr_scheduler.ReduceLROnPlateau):
                lr_scheduler.step(val_loss)
            else:
                lr_scheduler.step()

        # 调用epoch结束回调 / Call epoch-end callbacks
        if callbacks:
            for cb in callbacks:
                if hasattr(cb, 'on_epoch_end'):
                    cb.on_epoch_end(epoch, logs=history)
                if hasattr(cb, 'stop_training') and cb.stop_training:
                    stop_training = True
                    break

    # 调用训练结束回调 / Call train-end callbacks
    if callbacks:
        for cb in callbacks:
            if hasattr(cb, 'on_train_end'):
                cb.on_train_end(logs=history)

    return history


print("训练工具函数定义完成 / Training utility functions defined")

In [ ]:
class EarlyStopping:
    """
    早停回调 / Early stopping callback.

    当监控指标连续 patience 个 epoch 不改善时停止训练。
    等价于 Keras 的 keras.callbacks.EarlyStopping。

    Stops training when the monitored metric hasn't improved
    for patience consecutive epochs.
    Equivalent to Keras keras.callbacks.EarlyStopping.

    Parameters:
    -----------
    model : nn.Module
        待监控的模型 / Model to monitor
    monitor : str
        监控指标名 / Monitored metric name
    patience : int
        等待改善的epoch数 / Number of epochs to wait for improvement
    min_delta : float
        改善阈值 / Minimum change to qualify as improvement
    restore_best_weights : bool
        是否恢复最佳权重 / Whether to restore best weights on stop
    """

    def __init__(self, model, monitor='val_loss', patience=10, min_delta=0.0,
                 restore_best_weights=True):
        self.model = model
        self.monitor = monitor
        self.patience = patience
        self.min_delta = min_delta
        self.restore_best_weights = restore_best_weights
        self.best = float('inf')
        self.best_weights = None
        self.best_epoch = 0
        self.wait = 0
        self.stop_training = False

    def on_train_begin(self, logs=None):
        """训练开始时重置状态 / Reset state at training start."""
        self.wait = 0
        self.stop_training = False
        self.best = float('inf')

    def on_epoch_end(self, epoch, logs=None):
        """检查是否应该停止训练 / Check whether training should stop."""
        logs = logs or {}
        current = logs.get(self.monitor)
        if current is None:
            return

        if current < self.best - self.min_delta:
            self.best = current
            self.best_epoch = epoch
            self.wait = 0
            if self.restore_best_weights:
                self.best_weights = copy.deepcopy(self.model.state_dict())
        else:
            self.wait += 1
            if self.wait >= self.patience:
                self.stop_training = True
                print(f"  EarlyStopping: 连续 {self.patience} 个epoch没有改善，"
                      f"停止训练 / No improvement for {self.patience} epochs, stopping")

    def on_train_end(self, logs=None):
        """训练结束时恢复最佳权重 / Restore best weights at training end."""
        if self.restore_best_weights and self.best_weights is not None:
            self.model.load_state_dict(self.best_weights)
            print(f"  EarlyStopping: 恢复最佳权重 (epoch {self.best_epoch + 1}) / "
                  f"Restored best weights (epoch {self.best_epoch + 1})")


class ModelCheckpoint:
    """
    模型检查点回调 / Model checkpoint callback.

    在监控指标改善时保存模型 state_dict。
    等价于 Keras 的 keras.callbacks.ModelCheckpoint。

    Saves model state_dict when the monitored metric improves.
    Equivalent to Keras keras.callbacks.ModelCheckpoint.

    Parameters:
    -----------
    model : nn.Module
        待保存的模型 / Model to save
    filepath : str
        保存路径 / Save path
    monitor : str
        监控指标名 / Monitored metric name
    save_best_only : bool
        是否只保存最佳 / Whether to save only the best
    mode : str
        'min'或'max' / 'min' or 'max'
    """

    def __init__(self, model, filepath, monitor='val_loss',
                 save_best_only=True, mode='min'):
        self.model = model
        self.filepath = filepath
        self.monitor = monitor
        self.save_best_only = save_best_only
        self.mode = mode
        self.best = float('inf') if mode == 'min' else float('-inf')

    def on_train_begin(self, logs=None):
        """确保目录存在 / Ensure directory exists."""
        os.makedirs(os.path.dirname(self.filepath) if os.path.dirname(self.filepath) else '.',
                    exist_ok=True)

    def on_epoch_end(self, epoch, logs=None):
        """根据监控指标保存模型 / Save model based on monitored metric."""
        logs = logs or {}
        current = logs.get(self.monitor)
        if current is None:
            return

        is_improvement = (self.mode == 'min' and current < self.best) or \
                         (self.mode == 'max' and current > self.best)

        if is_improvement:
            self.best = current
            torch.save(self.model.state_dict(), self.filepath)
            print(f"  ModelCheckpoint: {self.monitor} 改善至 {current:.4f}，"
                  f"模型已保存 / Model saved")


print("回调函数定义完成 / Callbacks defined")

## 5. 手动超参数搜索

使用找到的学习率，系统地测试不同的超参数组合。

Keras 中可以使用 Keras Tuner 进行系统搜索，PyTorch 中我们手动实现随机搜索。
也可以使用第三方库如 Optuna 进行更高级的贝叶斯优化。

In [ ]:
def evaluate_hyperparameters(n_hidden, n_neurons, learning_rate,
                            activation='relu', epochs=30):
    """
    评估一组超参数的性能 / Evaluate performance of a set of hyperparameters.

    构建模型、训练并返回验证集上的性能指标。
    使用早停防止过拟合。

    Builds a model, trains it, and returns validation performance.
    Uses early stopping to prevent overfitting.

    Parameters:
    -----------
    n_hidden : int
        隐藏层数量 / Number of hidden layers
    n_neurons : int
        每层神经元数量 / Number of neurons per layer
    learning_rate : float
        学习率 / Learning rate
    activation : str
        激活函数 / Activation function
    epochs : int
        最大训练轮数 / Maximum training epochs

    Returns:
    --------
    dict : 包含模型、历史和验证损失的字典 / Dict with model, history, and val_loss
    """
    model, optimizer = build_model(
        n_hidden=n_hidden,
        n_neurons=n_neurons,
        learning_rate=learning_rate,
        activation=activation
    )
    criterion = nn.MSELoss()

    # 使用早停防止过拟合 / Use early stopping to prevent overfitting
    early_stop = EarlyStopping(
        model, monitor='val_loss', patience=5, restore_best_weights=True
    )

    history = fit(
        model, train_loader, valid_loader, criterion, optimizer,
        epochs=epochs, device=device, callbacks=[early_stop], verbose=False
    )

    val_loss, val_mae = evaluate(model, valid_loader, criterion, device)

    return {
        'model': model,
        'history': history,
        'val_loss': val_loss,
        'val_mae': val_mae,
        'params': {
            'n_hidden': n_hidden,
            'n_neurons': n_neurons,
            'learning_rate': learning_rate,
            'activation': activation
        }
    }


# 修改fit函数以支持verbose参数 / Modify fit to support verbose parameter
# (为了简洁，这里直接在evaluate_hyperparameters中用简化版训练循环)
# 实际上我们重写一个静默版的fit / Actually, let's write a silent version of fit

def fit_silent(model, train_loader, valid_loader, criterion, optimizer,
               epochs, device, callbacks=None, lr_scheduler=None):
    """
    静默训练循环（不打印进度）/ Silent training loop (no progress printing).
    """
    history = {
        'loss': [], 'val_loss': [],
        'mae': [], 'val_mae': [],
        'lr': []
    }

    if callbacks:
        for cb in callbacks:
            if hasattr(cb, 'on_train_begin'):
                cb.on_train_begin(logs=history)

    stop_training = False
    for epoch in range(epochs):
        if stop_training:
            break

        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_mae = evaluate(model, valid_loader, criterion, device)
        _, train_mae = evaluate(model, train_loader, criterion, device)
        current_lr = optimizer.param_groups[0]['lr']

        history['loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['mae'].append(train_mae)
        history['val_mae'].append(val_mae)
        history['lr'].append(current_lr)

        if lr_scheduler is not None:
            if isinstance(lr_scheduler, optim.lr_scheduler.ReduceLROnPlateau):
                lr_scheduler.step(val_loss)
            else:
                lr_scheduler.step()

        if callbacks:
            for cb in callbacks:
                if hasattr(cb, 'on_epoch_end'):
                    cb.on_epoch_end(epoch, logs=history)
                if hasattr(cb, 'stop_training') and cb.stop_training:
                    stop_training = True
                    break

    if callbacks:
        for cb in callbacks:
            if hasattr(cb, 'on_train_end'):
                cb.on_train_end(logs=history)

    return history


# 重写evaluate_hyperparameters使用静默版 / Rewrite using silent version
def evaluate_hyperparameters(n_hidden, n_neurons, learning_rate,
                            activation='relu', epochs=30):
    """
    评估一组超参数的性能 / Evaluate performance of a set of hyperparameters.

    Parameters:
    -----------
    n_hidden : int
        隐藏层数量 / Number of hidden layers
    n_neurons : int
        每层神经元数量 / Number of neurons per layer
    learning_rate : float
        学习率 / Learning rate
    activation : str
        激活函数 / Activation function
    epochs : int
        最大训练轮数 / Maximum training epochs

    Returns:
    --------
    dict : 包含模型、历史和验证损失的字典 / Dict with model, history, and val_loss
    """
    model, optimizer = build_model(
        n_hidden=n_hidden,
        n_neurons=n_neurons,
        learning_rate=learning_rate,
        activation=activation
    )
    criterion = nn.MSELoss()

    early_stop = EarlyStopping(
        model, monitor='val_loss', patience=5, restore_best_weights=True
    )

    history = fit_silent(
        model, train_loader, valid_loader, criterion, optimizer,
        epochs=epochs, device=device, callbacks=[early_stop]
    )

    val_loss, val_mae = evaluate(model, valid_loader, criterion, device)

    return {
        'model': model,
        'history': history,
        'val_loss': val_loss,
        'val_mae': val_mae,
        'params': {
            'n_hidden': n_hidden,
            'n_neurons': n_neurons,
            'learning_rate': learning_rate,
            'activation': activation
        }
    }


print("超参数评估函数定义完成 / Hyperparameter evaluation function defined")

In [ ]:
# 定义搜索空间（简化版本用于演示）/ Define search space (simplified for demo)
param_grid = {
    'n_hidden': [1, 2, 3],
    'n_neurons': [30, 50, 100],
    'learning_rate': [0.001, 0.01, 0.1],
    'activation': ['relu', 'tanh']
}

# 随机选择一些组合进行测试（随机搜索）/ Random search
n_trials = 5  # 为了演示只测试5组 / Only 5 trials for demo
results = []

print("开始超参数搜索... / Starting hyperparameter search...\n")

for i in range(n_trials):
    # 随机选择超参数 / Randomly select hyperparameters
    params = {
        'n_hidden': int(np.random.choice(param_grid['n_hidden'])),
        'n_neurons': int(np.random.choice(param_grid['n_neurons'])),
        'learning_rate': float(np.random.choice(param_grid['learning_rate'])),
        'activation': str(np.random.choice(param_grid['activation']))
    }

    print(f"Trial {i+1}/{n_trials}: {params}")

    result = evaluate_hyperparameters(**params, epochs=20)
    results.append(result)

    print(f"  验证损失 / val_loss: {result['val_loss']:.4f}\n")

# 找出最佳结果 / Find best result
best_result = min(results, key=lambda x: x['val_loss'])
print("=" * 50)
print(f"最佳超参数 / Best params: {best_result['params']}")
print(f"最佳验证损失 / Best val_loss: {best_result['val_loss']:.4f}")

In [ ]:
# 可视化搜索结果 / Visualize search results
fig, ax = plt.subplots(figsize=(10, 5))

val_losses = [r['val_loss'] for r in results]
labels = [f"Trial {i+1}" for i in range(len(results))]
colors = ['green' if loss == min(val_losses) else 'steelblue' for loss in val_losses]

bars = ax.bar(labels, val_losses, color=colors)
ax.set_ylabel('Validation Loss (MSE)')
ax.set_title('Hyperparameter Search Results')
ax.grid(True, alpha=0.3, axis='y')

# 添加数值标签 / Add value labels
for bar, loss in zip(bars, val_losses):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{loss:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 6. 1-Cycle学习率策略

1-Cycle策略是一种高效的训练方法：
1. 学习率先从低值上升到最大值（热身阶段）
2. 然后从最大值下降到极低值（冷却阶段）

### 优势
- 更快收敛
- 更好的泛化性能
- 不需要早停

### PyTorch 的巨大优势：内置 OneCycleLR

Keras 中需要自己实现 `OneCycleScheduler` 回调类（约50行代码），
而 PyTorch 内置了 `torch.optim.lr_scheduler.OneCycleLR`，只需一行即可创建！
这是 PyTorch 在学习率调度方面的一个重要优势。

In [ ]:
# 使用 PyTorch 内置的 OneCycleLR / Use PyTorch's built-in OneCycleLR
# 这是 PyTorch 相比 Keras 的一个重要优势！
# This is a major advantage of PyTorch over Keras!

model, optimizer = build_model(n_hidden=2, n_neurons=50, learning_rate=0.01)
criterion = nn.MSELoss()

# 计算总步数 / Calculate total steps
epochs = 30
steps_per_epoch = len(train_loader)
total_steps = steps_per_epoch * epochs

# 一行代码创建1-Cycle调度器！/ Create 1-Cycle scheduler in one line!
# Keras需要约50行自定义回调代码，PyTorch只需这一行
# Keras needs ~50 lines of custom callback code; PyTorch needs just this one line
one_cycle_scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=0.01,              # 最大学习率 / Maximum learning rate
    total_steps=total_steps,  # 总训练步数 / Total training steps
    pct_start=0.3,            # 热身阶段比例 / Warmup phase percentage
    anneal_strategy='cos',    # 退火策略：余弦 / Annealing strategy: cosine
    div_factor=25.0,          # 初始LR = max_lr / div_factor / Initial LR ratio
    final_div_factor=1000.0,  # 最终LR = max_lr / final_div_factor / Final LR ratio
)

print("OneCycleLR 调度器已创建 / OneCycleLR scheduler created")
print("  最大学习率 / max_lr: 0.01")
print(f"  初始学习率 / initial_lr: {0.01 / 25.0:.6f}")
print(f"  最终学习率 / final_lr: {0.01 / 1000.0:.6f}")
print(f"  热身步数 / warmup steps: {int(total_steps * 0.3)}")
print(f"  总步数 / total steps: {total_steps}")

### 重要：OneCycleLR 是 step-level 调度器

`OneCycleLR` 是按 **batch step** 更新的调度器，不是按 epoch 更新。
因此需要在每个 batch 的 `optimizer.step()` 之后调用 `scheduler.step()`，
而不是在每个 epoch 结束时调用。

这意味着我们需要修改训练循环，将 `scheduler.step()` 放在 batch 级别。

In [ ]:
def train_one_epoch_with_scheduler(model, train_loader, criterion, optimizer,
                                   scheduler, device):
    """
    带step级调度器的单epoch训练 / Train one epoch with step-level scheduler.

    OneCycleLR 需要在每个 batch 后调用 scheduler.step()，
    因此不能使用 epoch 级的训练循环。

    OneCycleLR requires scheduler.step() after each batch,
    so we cannot use an epoch-level training loop.

    Parameters:
    -----------
    model : nn.Module
        PyTorch模型 / PyTorch model
    train_loader : DataLoader
        训练数据加载器 / Training data loader
    criterion : loss function
        损失函数 / Loss function
    optimizer : optim.Optimizer
        优化器 / Optimizer
    scheduler : lr_scheduler
        步级学习率调度器 / Step-level learning rate scheduler
    device : torch.device
        计算设备 / Compute device

    Returns:
    --------
    float : 平均训练损失 / Average training loss
    """
    model.train()
    total_loss = 0.0
    total_samples = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch)
        loss.backward()
        optimizer.step()
        # OneCycleLR: 在每个batch后更新学习率 / Update LR after each batch
        scheduler.step()
        total_loss += loss.item() * X_batch.size(0)
        total_samples += X_batch.size(0)
    return total_loss / total_samples


def fit_with_onecycle(model, train_loader, valid_loader, criterion, optimizer,
                      scheduler, epochs, device, callbacks=None):
    """
    使用OneCycleLR的训练循环 / Training loop with OneCycleLR scheduler.

    OneCycleLR 是 step-level 调度器，需要在每个 batch 后调用 scheduler.step()。
    此函数专门为 OneCycleLR 设计，将调度器集成到 batch 级训练循环中。

    OneCycleLR is a step-level scheduler that requires scheduler.step()
    after each batch. This function is specifically designed for OneCycleLR,
    integrating the scheduler into the batch-level training loop.

    Parameters:
    -----------
    model : nn.Module
        PyTorch模型 / PyTorch model
    train_loader : DataLoader
        训练数据加载器 / Training data loader
    valid_loader : DataLoader
        验证数据加载器 / Validation data loader
    criterion : loss function
        损失函数 / Loss function
    optimizer : optim.Optimizer
        优化器 / Optimizer
    scheduler : OneCycleLR
        OneCycleLR调度器 / OneCycleLR scheduler
    epochs : int
        训练轮数 / Number of epochs
    device : torch.device
        计算设备 / Compute device
    callbacks : list, optional
        回调列表 / List of callbacks

    Returns:
    --------
    dict : 训练历史 / Training history
    """
    history = {
        'loss': [], 'val_loss': [],
        'mae': [], 'val_mae': [],
        'lr': []
    }

    if callbacks:
        for cb in callbacks:
            if hasattr(cb, 'on_train_begin'):
                cb.on_train_begin(logs=history)

    stop_training = False
    for epoch in range(epochs):
        if stop_training:
            break

        # 使用带调度器的训练函数 / Train with scheduler
        train_loss = train_one_epoch_with_scheduler(
            model, train_loader, criterion, optimizer, scheduler, device
        )

        # 评估 / Evaluate
        val_loss, val_mae = evaluate(model, valid_loader, criterion, device)
        _, train_mae = evaluate(model, train_loader, criterion, device)

        # 记录当前学习率 / Record current learning rate
        current_lr = optimizer.param_groups[0]['lr']

        history['loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['mae'].append(train_mae)
        history['val_mae'].append(val_mae)
        history['lr'].append(current_lr)

        print(f"Epoch {epoch+1}/{epochs} - "
              f"loss: {train_loss:.4f} - mae: {train_mae:.4f} - "
              f"val_loss: {val_loss:.4f} - val_mae: {val_mae:.4f} - "
              f"lr: {current_lr:.6f}")

        if callbacks:
            for cb in callbacks:
                if hasattr(cb, 'on_epoch_end'):
                    cb.on_epoch_end(epoch, logs=history)
                if hasattr(cb, 'stop_training') and cb.stop_training:
                    stop_training = True
                    break

    if callbacks:
        for cb in callbacks:
            if hasattr(cb, 'on_train_end'):
                cb.on_train_end(logs=history)

    return history


print("OneCycleLR 训练函数定义完成 / OneCycleLR training functions defined")

In [ ]:
# 使用1-Cycle策略训练模型 / Train model with 1-Cycle strategy
model, optimizer = build_model(n_hidden=2, n_neurons=50, learning_rate=0.01)
criterion = nn.MSELoss()

epochs = 30
steps_per_epoch = len(train_loader)
total_steps = steps_per_epoch * epochs

# 创建OneCycleLR调度器 / Create OneCycleLR scheduler
one_cycle_scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=0.01,
    total_steps=total_steps,
    pct_start=0.3,
    anneal_strategy='cos',
    div_factor=25.0,
    final_div_factor=1000.0,
)

# 训练 / Train
history = fit_with_onecycle(
    model, train_loader, valid_loader, criterion, optimizer,
    one_cycle_scheduler, epochs=epochs, device=device
)

In [ ]:
# 绘制学习率曲线 / Plot learning rate curve
# OneCycleLR 的学习率变化是 step 级别的，我们记录的是 epoch 级别的
# 为了更精确地展示，我们重新生成 step 级别的学习率曲线

# 生成 step 级别的学习率曲线 / Generate step-level LR curve
lrs_step = []
temp_optimizer = optim.SGD([torch.zeros(1, requires_grad=True)], lr=0.01)
temp_scheduler = optim.lr_scheduler.OneCycleLR(
    temp_optimizer, max_lr=0.01, total_steps=total_steps,
    pct_start=0.3, anneal_strategy='cos',
    div_factor=25.0, final_div_factor=1000.0
)
for step in range(total_steps):
    lrs_step.append(temp_optimizer.param_groups[0]['lr'])
    temp_optimizer.step()
    temp_scheduler.step()

plt.figure(figsize=(10, 4))
plt.plot(lrs_step)
plt.xlabel('Step')
plt.ylabel('Learning Rate')
plt.title('1-Cycle Learning Rate Schedule (OneCycleLR)')
warmup_steps = int(total_steps * 0.3)
plt.axvline(x=warmup_steps, color='r', linestyle='--',
            label=f'Warmup End ({warmup_steps} steps)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# 绘制训练曲线 / Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 损失曲线 / Loss curves
axes[0].plot(history['loss'], label='Training')
axes[0].plot(history['val_loss'], label='Validation')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (MSE)')
axes[0].set_title('Loss Curves (1-Cycle)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MAE曲线 / MAE curves
axes[1].plot(history['mae'], label='Training')
axes[1].plot(history['val_mae'], label='Validation')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].set_title('MAE Curves (1-Cycle)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 最终评估 / Final evaluation
test_loss, test_mae = evaluate(model, test_loader, criterion, device)
print(f"\n测试集 / Test set - MSE: {test_loss:.4f}, MAE: {test_mae:.4f}")

## 7. 对比实验：固定学习率 vs 1-Cycle

In [ ]:
# 使用固定学习率训练 / Train with fixed learning rate
model_fixed, optimizer_fixed = build_model(n_hidden=2, n_neurons=50, learning_rate=0.01)
criterion = nn.MSELoss()

history_fixed = fit_silent(
    model_fixed, train_loader, valid_loader, criterion, optimizer_fixed,
    epochs=30, device=device
)

# 使用1-Cycle策略训练 / Train with 1-Cycle strategy
model_1cycle, optimizer_1cycle = build_model(n_hidden=2, n_neurons=50, learning_rate=0.01)

one_cycle_scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer_1cycle,
    max_lr=0.01,
    total_steps=steps_per_epoch * 30,
    pct_start=0.3,
    anneal_strategy='cos',
    div_factor=25.0,
    final_div_factor=1000.0,
)

history_1cycle = fit_with_onecycle(
    model_1cycle, train_loader, valid_loader, criterion, optimizer_1cycle,
    one_cycle_scheduler, epochs=30, device=device
)

# 对比结果 / Compare results
plt.figure(figsize=(10, 5))
plt.plot(history_fixed['val_loss'], label='Fixed LR', linestyle='--')
plt.plot(history_1cycle['val_loss'], label='1-Cycle (OneCycleLR)')
plt.xlabel('Epoch')
plt.ylabel('Validation Loss')
plt.title('Fixed LR vs 1-Cycle Learning Rate')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# 最终对比 / Final comparison
fixed_loss, _ = evaluate(model_fixed, test_loader, criterion, device)
cycle_loss, _ = evaluate(model_1cycle, test_loader, criterion, device)

print(f"固定学习率测试MSE / Fixed LR test MSE: {fixed_loss:.4f}")
print(f"1-Cycle测试MSE / 1-Cycle test MSE: {cycle_loss:.4f}")
improvement = (fixed_loss - cycle_loss) / fixed_loss * 100
print(f"改进 / Improvement: {improvement:.1f}%")

## 8. 使用最佳超参数训练最终模型

In [ ]:
# 使用找到的最佳超参数 / Use the best hyperparameters found
best_params = best_result['params']
print(f"最佳超参数 / Best params: {best_params}")

# 构建最终模型 / Build final model
final_model, final_optimizer = build_model(
    n_hidden=best_params['n_hidden'],
    n_neurons=best_params['n_neurons'],
    learning_rate=best_params['learning_rate'],
    activation=best_params['activation']
)
criterion = nn.MSELoss()

# 计算1-Cycle参数 / Calculate 1-Cycle parameters
epochs = 50
total_steps = steps_per_epoch * epochs

# 使用OneCycleLR策略 / Use OneCycleLR strategy
one_cycle_scheduler = optim.lr_scheduler.OneCycleLR(
    final_optimizer,
    max_lr=best_params['learning_rate'],
    total_steps=total_steps,
    pct_start=0.3,
    anneal_strategy='cos',
    div_factor=25.0,
    final_div_factor=1000.0,
)

# 添加模型检查点 / Add model checkpoint
checkpoint_cb = ModelCheckpoint(
    final_model,
    filepath='checkpoints_hypertune/best_model.pth',
    monitor='val_loss',
    save_best_only=True,
    mode='min'
)

# 训练 / Train
history = fit_with_onecycle(
    final_model, train_loader, valid_loader, criterion, final_optimizer,
    one_cycle_scheduler, epochs=epochs, device=device,
    callbacks=[checkpoint_cb]
)

# 加载最佳模型 / Load best model
best_model, _ = build_model(
    n_hidden=best_params['n_hidden'],
    n_neurons=best_params['n_neurons'],
    learning_rate=best_params['learning_rate'],
    activation=best_params['activation']
)
best_model.load_state_dict(torch.load('checkpoints_hypertune/best_model.pth', weights_only=True))
best_model.to(device)

# 最终评估 / Final evaluation
test_loss, test_mae = evaluate(best_model, test_loader, criterion, device)
print(f"\n最终模型测试集 / Final model on test set - MSE: {test_loss:.4f}, MAE: {test_mae:.4f}")

## 9. TF vs PyTorch 对照

### 超参数调整对照表

| 功能 | TensorFlow / Keras | PyTorch |
|------|-------------------|---------|
| **模型构建函数** | `build_model()` 返回编译好的 `keras.Model` | `build_model()` 返回 `(nn.Module, optimizer)` 元组 |
| **LR Finder** | `LRFinder(keras.callbacks.Callback)` 回调 | 手动实现训练循环，通过 `optimizer.param_groups[0]['lr']` 读写学习率 |
| **超参数搜索** | Keras Tuner（内置） | 手动随机搜索 或 Optuna（第三方） |
| **1-Cycle策略** | 自定义 `OneCycleScheduler` 回调（约50行代码） | `torch.optim.lr_scheduler.OneCycleLR`（一行代码！） |
| **早停** | `keras.callbacks.EarlyStopping` | 自定义 `EarlyStopping` 类 |
| **模型检查点** | `keras.callbacks.ModelCheckpoint` 保存完整模型 | 自定义 `ModelCheckpoint` 保存 `state_dict` |
| **学习率读取** | `model.optimizer.learning_rate` | `optimizer.param_groups[0]['lr']` |
| **学习率设置** | `optimizer.learning_rate.assign(lr)` | `optimizer.param_groups[0]['lr'] = lr` |

### 学习率调度器对照表

| Keras | PyTorch | 说明 |
|-------|---------|------|
| 自定义 `OneCycleScheduler` 回调 | **`OneCycleLR`** (内置) | **PyTorch 优势！** Keras 需要约50行自定义代码 |
| `ReduceLROnPlateau` 回调 | `ReduceLROnPlateau` 调度器 | 功能等价，PyTorch版需传 `val_loss` |
| `LearningRateScheduler(fn)` 回调 | `LambdaLR(optimizer, lr_lambda=fn)` | 自定义函数调度 |
| 无直接等价 | `CosineAnnealingLR` | 余弦退火 |
| 无直接等价 | `CyclicLR` | 周期性学习率 |

### 关键差异

1. **OneCycleLR 是 PyTorch 的重大优势**：Keras 没有内置1-Cycle调度器，需要自己实现回调（约50行代码）；
   PyTorch 的 `OneCycleLR` 只需一行代码即可创建，且支持余弦和线性两种退火策略

2. **LR Finder 实现方式不同**：Keras 通过回调自动集成到 `model.fit()`；
   PyTorch 需要手动实现训练循环，但这也让我们更深入地理解了 LR Finder 的工作原理

3. **超参数搜索工具**：Keras 有 Keras Tuner（官方工具）；
   PyTorch 没有官方工具，但可以使用 Optuna、Ray Tune 等第三方库

4. **学习率读写方式**：Keras 使用 `optimizer.learning_rate` 属性；
   PyTorch 使用 `optimizer.param_groups[0]['lr']`，支持不同参数组使用不同学习率（更灵活）

5. **调度器更新粒度**：Keras 回调在 epoch 或 batch 级别自动触发；
   PyTorch 需要手动调用 `scheduler.step()`，且 `OneCycleLR` 是 step-level 调度器，
   必须在每个 batch 后调用

## 练习

### 练习1：使用 Optuna 进行贝叶斯超参数优化
安装 Optuna (`pip install optuna`)，将本教程中的手动随机搜索替换为 Optuna 的贝叶斯优化：
- 定义 `objective(trial)` 函数，使用 `trial.suggest_int()` 和 `trial.suggest_float()` 定义搜索空间
- 使用 `study = optuna.create_study(direction='minimize')` 创建研究
- 运行 `study.optimize(objective, n_trials=20)` 并分析结果
- 绘制 Optuna 的参数重要性和优化历史图

思考：贝叶斯优化相比随机搜索有什么优势？在什么情况下随机搜索可能更合适？

### 练习2：对比 OneCycleLR 的两种退火策略
`OneCycleLR` 支持 `anneal_strategy='cos'`（余弦退火）和 `anneal_strategy='linear'`（线性退火）两种策略。
使用相同的模型和数据，分别用两种策略训练，对比：
- 学习率曲线的形状差异
- 训练和验证损失曲线
- 最终测试集性能

提示：只需将 `anneal_strategy='cos'` 改为 `anneal_strategy='linear'` 即可。

### 练习3：实现带动量调度的1-Cycle策略
Leslie Smith 的原始1-Cycle论文中，学习率和动量是反向调度的：
- 学习率上升时，动量下降
- 学习率下降时，动量上升

在 PyTorch 的 `OneCycleLR` 中，可以通过 `cycle_momentum=True` 参数启用此功能
（仅支持 SGD 等带有 momentum 参数的优化器）。

尝试以下实验：
1. 使用 SGD + `cycle_momentum=True` 训练模型
2. 使用 SGD + `cycle_momentum=False` 训练模型
3. 对比两者的训练曲线和最终性能

提示：`OneCycleLR` 的 `base_momentum` 和 `max_momentum` 参数控制动量范围。

## 小结

### 超参数调整流程

1. **使用LR Finder**: 确定学习率的合适范围
2. **随机搜索**: 在参数空间中随机采样，比网格搜索更高效
3. **1-Cycle策略**: 使用 `OneCycleLR` 调度器提升训练效果
4. **早停和检查点**: 防止过拟合并保存最佳模型

### 关键超参数优先级

1. **学习率**: 最重要，使用LR Finder确定
2. **网络深度**: 从浅层开始，逐步增加
3. **神经元数量**: 宁可多一些，用正则化控制
4. **批量大小**: 32-64通常是好的起点

### PyTorch 特有要点

- `OneCycleLR` 是内置调度器，无需自己实现1-Cycle策略
- `OneCycleLR` 是 step-level 调度器，必须在每个 batch 后调用 `scheduler.step()`
- `optimizer.param_groups[0]['lr']` 是读写学习率的标准方式
- LR Finder 需要手动实现训练循环，但有助于深入理解原理

### 进阶工具

- **Optuna**: 贝叶斯优化，比随机搜索更高效
- **Ray Tune**: 分布式超参数搜索
- **Weights & Biases**: 实验跟踪和可视化